# Detección de Blanqueo de Capitales (AML)

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Detectar patrones de blanqueo** con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Identificar structuring** (fraccionamiento de operaciones bajo umbrales regulatorios)
3. **Analizar redes de transferencias** sospechosas entre cuentas
4. **Calcular indicadores de riesgo AML** con window functions
5. **Generar alertas automáticas** con clasificación por severidad

---

## Lo Que Construirás

Un **sistema de detección AML** que identifica actividad sospechosa:
- Detección de structuring (operaciones justo bajo 3.000€ y 10.000€)
- Análisis de velocidad de transferencias (volumen inusual en período corto)
- Red de beneficiarios sospechosos
- Clasificación de alertas por severidad
- Dashboard de investigación

**Valor de Negocio**: Cumplir con regulaciones de prevención de blanqueo (6AMLD, SEPBLAC) y evitar multas millonarias.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex ML habilitado
- **Aproximadamente 15 minutos** para completar
- **100% SQL** (excepto dashboard Streamlit)

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Modelo de detección de actividad sospechosa
- Window functions — Análisis temporal de patrones
- `COUNT() OVER` — Frecuencia de transacciones en ventanas temporales
- `SUM() OVER` — Volumen acumulado por período
- `GENERATOR()` — Datos sintéticos de operaciones bancarias

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Anti-Money Laundering

**Objetivo**: Detectar automáticamente operaciones que podrían constituir blanqueo de capitales.

### Umbrales Regulatorios

- **3.000€**: Umbral de identificación en pagos en efectivo (España)
- **10.000€**: Umbral de comunicación obligatoria al SEPBLAC
- **Structuring**: Fraccionar operaciones para evitar estos umbrales

### Patrones Sospechosos

- Múltiples transferencias justo bajo el umbral
- Alta velocidad de transacciones en períodos cortos
- Beneficiarios en jurisdicciones de alto riesgo
- Inconsistencia entre perfil del cliente y volumen de operaciones

In [ ]:
CREATE DATABASE IF NOT EXISTS AML_DETECCION_DB;
CREATE SCHEMA IF NOT EXISTS AML_DETECCION_DB.PUBLIC;
USE SCHEMA AML_DETECCION_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

---

## Paso 2: Definir Estructura de Datos

### Tablas

1. **`CUENTAS`** — Cuentas bancarias con perfil de titular
2. **`OPERACIONES`** — Transacciones con origen y destino
3. **`BENEFICIARIOS`** — Destinos de transferencias con jurisdicción

In [ ]:
CREATE OR REPLACE TABLE CUENTAS (
    cuenta_id VARCHAR(15) PRIMARY KEY,
    titular VARCHAR(100),
    tipo_cuenta VARCHAR(20),
    fecha_apertura DATE,
    perfil_riesgo VARCHAR(10),
    ingresos_declarados DECIMAL(12,2),
    pais_residencia VARCHAR(30)
);

CREATE OR REPLACE TABLE OPERACIONES (
    operacion_id VARCHAR(12) PRIMARY KEY,
    cuenta_origen VARCHAR(15),
    cuenta_destino VARCHAR(15),
    fecha_hora TIMESTAMP,
    importe DECIMAL(12,2),
    tipo_operacion VARCHAR(30),
    canal VARCHAR(20),
    pais_destino VARCHAR(30),
    es_sospechosa BOOLEAN,
    FOREIGN KEY (cuenta_origen) REFERENCES CUENTAS(cuenta_id)
);

CREATE OR REPLACE TABLE BENEFICIARIOS (
    beneficiario_id VARCHAR(10) PRIMARY KEY,
    nombre VARCHAR(100),
    pais VARCHAR(30),
    nivel_riesgo_jurisdiccion VARCHAR(10),
    num_operaciones_recibidas INTEGER
);

SELECT 'Tablas creadas' AS status;

---

## Paso 3: Generar Datos Sintéticos

### Qué Vamos a Crear

- **3.000 cuentas** con diferentes perfiles de riesgo
- **80.000 operaciones** (3% sospechosas)
- **500 beneficiarios** en jurisdicciones de diferente riesgo

### Patrones de Blanqueo Simulados

- **Structuring**: Operaciones de 2.800-2.999€ (justo bajo umbral de 3.000€)
- **Smurfing**: Múltiples operaciones pequeñas a diferentes beneficiarios
- **Layering**: Cadenas de transferencias entre cuentas

In [ ]:
-- Generar cuentas
INSERT INTO CUENTAS
SELECT
    'ES' || LPAD(SEQ4()::STRING, 12, '0'),
    'TITULAR_' || SEQ4(),
    ARRAY_CONSTRUCT('Corriente','Ahorro','Empresa','Inversión')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    DATEADD('day', -UNIFORM(30, 3650, RANDOM()), CURRENT_DATE()),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 5 THEN 'Alto'
         WHEN UNIFORM(0,100,RANDOM()) < 20 THEN 'Medio'
         ELSE 'Bajo' END,
    ROUND(UNIFORM(12000, 200000, RANDOM()), 2),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 90 THEN 'España'
         ELSE ARRAY_CONSTRUCT('Portugal','Francia','Marruecos','Colombia','China')[UNIFORM(0,4,RANDOM())]::VARCHAR END
FROM TABLE(GENERATOR(ROWCOUNT => 3000));

-- Generar beneficiarios
INSERT INTO BENEFICIARIOS
SELECT
    'BEN' || LPAD(SEQ4()::STRING, 5, '0'),
    'BENEFICIARIO_' || SEQ4(),
    ARRAY_CONSTRUCT('España','Portugal','Francia','Alemania','Panamá','Islas Caimán','Emiratos Árabes','Suiza','Hong Kong','Nigeria')[UNIFORM(0,9,RANDOM())]::VARCHAR,
    CASE WHEN UNIFORM(0,9,RANDOM()) >= 6 THEN 'Alto' 
         WHEN UNIFORM(0,9,RANDOM()) >= 3 THEN 'Medio'
         ELSE 'Bajo' END,
    UNIFORM(1, 200, RANDOM())
FROM TABLE(GENERATOR(ROWCOUNT => 500));

-- Generar operaciones (3% sospechosas)
INSERT INTO OPERACIONES
WITH base AS (
    SELECT
        'OP' || LPAD(SEQ4()::STRING, 8, '0') AS operacion_id,
        'ES' || LPAD(UNIFORM(1,3000,RANDOM())::STRING, 12, '0') AS cuenta_origen,
        'ES' || LPAD(UNIFORM(1,3000,RANDOM())::STRING, 12, '0') AS cuenta_destino,
        DATEADD('minute', -UNIFORM(0, 525600, RANDOM()), CURRENT_TIMESTAMP()) AS fecha_hora,
        CASE WHEN UNIFORM(0,100,RANDOM()) < 3 THEN TRUE ELSE FALSE END AS es_sosp
    FROM TABLE(GENERATOR(ROWCOUNT => 80000))
)
SELECT
    operacion_id, cuenta_origen, cuenta_destino, fecha_hora,
    CASE WHEN es_sosp THEN ROUND(UNIFORM(2800, 2999, RANDOM()) + UNIFORM(0,99,RANDOM())/100, 2)
         ELSE ROUND(UNIFORM(10, 5000, RANDOM()), 2) END AS importe,
    ARRAY_CONSTRUCT('Transferencia','Ingreso efectivo','Pago','Traspaso','Domiciliación')[UNIFORM(0,4,RANDOM())]::VARCHAR,
    ARRAY_CONSTRUCT('Online','Oficina','ATM','Banca telefónica')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    CASE WHEN es_sosp THEN ARRAY_CONSTRUCT('Panamá','Islas Caimán','Emiratos','Nigeria','Suiza')[UNIFORM(0,4,RANDOM())]::VARCHAR
         ELSE ARRAY_CONSTRUCT('España','España','España','Portugal','Francia','Alemania')[UNIFORM(0,5,RANDOM())]::VARCHAR END,
    es_sosp
FROM base;

SELECT es_sospechosa, COUNT(*), ROUND(AVG(importe),2) AS importe_medio FROM OPERACIONES GROUP BY 1;

---

## Paso 4: Ingeniería de Variables AML

### Indicadores de Riesgo

- **Operaciones bajo umbral**: Nº de operaciones entre 2.800€ y 3.000€
- **Velocidad transaccional**: Nº de operaciones en últimas 24h/7d
- **Concentración destinos**: % de operaciones a países de alto riesgo
- **Ratio volumen/ingresos**: Volumen total vs ingresos declarados

In [ ]:
-- Feature engineering AML
CREATE OR REPLACE TABLE FEATURES_AML AS
WITH ops_por_cuenta AS (
    SELECT
        cuenta_origen,
        COUNT(*) AS total_operaciones,
        SUM(importe) AS volumen_total,
        AVG(importe) AS importe_medio,
        SUM(CASE WHEN importe BETWEEN 2800 AND 2999.99 THEN 1 ELSE 0 END) AS ops_bajo_umbral_3k,
        SUM(CASE WHEN importe BETWEEN 9000 AND 9999.99 THEN 1 ELSE 0 END) AS ops_bajo_umbral_10k,
        SUM(CASE WHEN pais_destino IN ('Panamá','Islas Caimán','Emiratos','Nigeria') THEN 1 ELSE 0 END) AS ops_pais_riesgo,
        COUNT(DISTINCT pais_destino) AS paises_destino_distintos,
        COUNT(DISTINCT cuenta_destino) AS beneficiarios_distintos,
        MAX(importe) AS importe_maximo
    FROM OPERACIONES GROUP BY 1
)
SELECT
    o.operacion_id,
    o.importe::FLOAT,
    CASE WHEN o.importe BETWEEN 2800 AND 2999.99 THEN 1 ELSE 0 END::FLOAT AS es_structuring,
    CASE WHEN o.pais_destino IN ('Panamá','Islas Caimán','Emiratos','Nigeria') THEN 1 ELSE 0 END::FLOAT AS destino_alto_riesgo,
    opc.total_operaciones::FLOAT,
    opc.volumen_total::FLOAT,
    opc.ops_bajo_umbral_3k::FLOAT,
    opc.ops_bajo_umbral_10k::FLOAT,
    opc.ops_pais_riesgo::FLOAT,
    opc.paises_destino_distintos::FLOAT,
    opc.beneficiarios_distintos::FLOAT,
    (opc.volumen_total / NULLIF(c.ingresos_declarados, 0))::FLOAT AS ratio_vol_ingresos,
    CASE WHEN c.perfil_riesgo = 'Alto' THEN 2 WHEN c.perfil_riesgo = 'Medio' THEN 1 ELSE 0 END::FLOAT AS nivel_riesgo_cliente,
    o.es_sospechosa
FROM OPERACIONES o
JOIN ops_por_cuenta opc ON o.cuenta_origen = opc.cuenta_origen
JOIN CUENTAS c ON o.cuenta_origen = c.cuenta_id;

SELECT * FROM FEATURES_AML LIMIT 10;

---

## Paso 5: Entrenar Modelo de Detección

### Modelo AML

El clasificador aprenderá a identificar operaciones sospechosas basándose en los patrones de structuring, velocidad y concentración geográfica.

In [ ]:
-- Dividir y entrenar
CREATE OR REPLACE TABLE TRAIN_AML AS SELECT * FROM FEATURES_AML SAMPLE (80);
CREATE OR REPLACE TABLE TEST_AML AS SELECT * FROM FEATURES_AML WHERE operacion_id NOT IN (SELECT operacion_id FROM TRAIN_AML);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_aml(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_AML'),
    TARGET_COLNAME => 'ES_SOSPECHOSA'
);

---

## Paso 6: Evaluar y Generar Alertas

### Niveles de Alerta

- **CRÍTICA** (≥80%): Reporte inmediato al SEPBLAC
- **ALTA** (≥60%): Investigación prioritaria
- **MEDIA** (≥40%): Revisión programada
- **BAJA** (<40%): Monitorización continua

In [ ]:
CALL detector_aml!SHOW_EVALUATION_METRICS();
CALL detector_aml!SHOW_FEATURE_IMPORTANCE();

In [ ]:
-- Generar alertas con severidad
CREATE OR REPLACE TABLE ALERTAS_AML AS
SELECT
    operacion_id, importe, es_structuring, destino_alto_riesgo,
    ratio_vol_ingresos, es_sospechosa AS sospechosa_real,
    detector_aml!PREDICT(
        OBJECT_CONSTRUCT(
            'IMPORTE', importe, 'ES_STRUCTURING', es_structuring,
            'DESTINO_ALTO_RIESGO', destino_alto_riesgo,
            'TOTAL_OPERACIONES', total_operaciones,
            'VOLUMEN_TOTAL', volumen_total,
            'OPS_BAJO_UMBRAL_3K', ops_bajo_umbral_3k,
            'OPS_BAJO_UMBRAL_10K', ops_bajo_umbral_10k,
            'OPS_PAIS_RIESGO', ops_pais_riesgo,
            'PAISES_DESTINO_DISTINTOS', paises_destino_distintos,
            'BENEFICIARIOS_DISTINTOS', beneficiarios_distintos,
            'RATIO_VOL_INGRESOS', ratio_vol_ingresos,
            'NIVEL_RIESGO_CLIENTE', nivel_riesgo_cliente
        )
    ) AS pred,
    ROUND(pred['probability']['true']::FLOAT * 100, 1) AS prob_sospechosa,
    CASE
        WHEN pred['probability']['true']::FLOAT >= 0.8 THEN 'CRITICA'
        WHEN pred['probability']['true']::FLOAT >= 0.6 THEN 'ALTA'
        WHEN pred['probability']['true']::FLOAT >= 0.4 THEN 'MEDIA'
        ELSE 'BAJA'
    END AS severidad
FROM TEST_AML;

SELECT severidad, COUNT(*) AS n, ROUND(AVG(prob_sospechosa),1) AS prob_media
FROM ALERTAS_AML GROUP BY 1 ORDER BY prob_media DESC;

---

## Paso 7: Dashboard de Investigación

### Panel AML

Dashboard para el equipo de compliance con alertas filtradas por severidad.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Detección de Blanqueo de Capitales (AML)')
st.markdown('### Panel de Alertas')

total = session.sql('SELECT COUNT(*) FROM ALERTAS_AML').collect()[0][0]
criticas = session.sql("SELECT COUNT(*) FROM ALERTAS_AML WHERE severidad='CRITICA'").collect()[0][0]
altas = session.sql("SELECT COUNT(*) FROM ALERTAS_AML WHERE severidad='ALTA'").collect()[0][0]

c1, c2, c3, c4 = st.columns(4)
c1.metric('Total Operaciones', f'{total:,}')
c2.metric('Alertas Críticas', f'{criticas:,}')
c3.metric('Alertas Altas', f'{altas:,}')
c4.metric('Tasa Detección', f'{(criticas+altas)/total*100:.1f}%')

st.markdown('---')
st.subheader('Distribución de Alertas')
df = session.sql('SELECT severidad, COUNT(*) AS n FROM ALERTAS_AML GROUP BY 1 ORDER BY n DESC').to_pandas()
st.bar_chart(df.set_index('SEVERIDAD'))

st.markdown('---')
st.subheader('Operaciones para Investigar')
sev = st.selectbox('Severidad', ['CRITICA','ALTA','MEDIA','BAJA'])
df_ops = session.sql(f"SELECT operacion_id, importe, prob_sospechosa, es_structuring, destino_alto_riesgo FROM ALERTAS_AML WHERE severidad='{sev}' ORDER BY prob_sospechosa DESC LIMIT 30").to_pandas()
st.dataframe(df_ops)

---

## Paso 8: Limpieza del Entorno (Opcional)

⚠️ **Advertencia**: Eliminará todos los datos y modelos.

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS AML_DETECCION_DB;